# 09 Full Experiment Suite Runner

Run the complete experiment suite (all core methods, sweeps, and report generation) from inside a notebook.

In [ ]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


In [ ]:
from pathlib import Path
import os
import time

from cytof_archetypes.experiments import load_suite_config, run_experiment_suite

CONFIG_PATH = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
assert CONFIG_PATH.exists(), f'Missing config: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH)

# Optional: override output dir without editing YAML.
# os.environ['CYTOF_SUITE_OUTPUT_DIR'] = str(REPO_ROOT / 'outputs' / 'experiment_suite')

In [ ]:
cfg = load_suite_config(CONFIG_PATH)
env_out = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
if env_out:
    cfg['output_dir'] = env_out

# Speed controls for quick testing:
DOWNSAMPLE_FACTOR = 1  # e.g. 5 keeps ~20% of cells, 10 keeps ~10%
if DOWNSAMPLE_FACTOR and DOWNSAMPLE_FACTOR > 1:
    cfg.setdefault('dataset', {})['downsample_factor'] = int(DOWNSAMPLE_FACTOR)
else:
    cfg.setdefault('dataset', {}).pop('downsample_factor', None)
    cfg.setdefault('dataset', {}).pop('downsample_fraction', None)

# Verbose execution prints and progress bars:
cfg['show_progress'] = True
cfg['show_run_logs'] = True
NN_TRAINING_PROGRESS = True
NN_TRAINING_PROGRESS_LEVEL = 'epoch'  # 'epoch' or 'batch'
cfg['show_training_progress'] = bool(NN_TRAINING_PROGRESS)
cfg['training_progress_level'] = str(NN_TRAINING_PROGRESS_LEVEL)
cfg['training_progress_leave'] = False
CPU_MULTIPROCESS_WORKERS = 1  # >1 parallelizes CPU baselines (nmf/classical) across processes
cfg['cpu_multiprocessing_workers'] = int(CPU_MULTIPROCESS_WORKERS)
cfg['cpu_parallel_methods'] = ['nmf', 'classical_archetypes']

print('Resolved output_dir:', cfg['output_dir'])
print('Resolved default device:', cfg.get('resolved_device', cfg.get('device')))
for _m in ['deterministic_archetypal_ae', 'probabilistic_archetypal_ae', 'ae', 'vae']:
    print(f"  {_m}:", cfg.get('methods', {}).get(_m, {}).get('device'))
print('Methods:', list(cfg.get('methods', {}).keys()))
print('Seeds:', cfg.get('seeds', []))
print('K sweep:', cfg.get('sweeps', {}).get('k_values', []))
print('Latent sweep:', cfg.get('sweeps', {}).get('latent_dims', []))
print('Downsample factor:', cfg.get('dataset', {}).get('downsample_factor', 1))
print('show_progress:', cfg.get('show_progress'), 'show_run_logs:', cfg.get('show_run_logs'))
print('show_training_progress:', cfg.get('show_training_progress'), 'training_progress_level:', cfg.get('training_progress_level'))
print('cpu_multiprocessing_workers:', cfg.get('cpu_multiprocessing_workers'), 'cpu_parallel_methods:', cfg.get('cpu_parallel_methods'))

In [ ]:
RUN_FULL = True  # set to False to skip execution
if RUN_FULL:
    t0 = time.time()
    out_dir = run_experiment_suite(cfg)
    dt = time.time() - t0
    print(f'Completed full suite in {dt/60:.2f} minutes')
    print('Output directory:', out_dir)
else:
    out_dir = Path(cfg['output_dir'])
    print('Execution skipped. Expected output directory:', out_dir)

In [ ]:
import pandas as pd

out_dir = Path(cfg['output_dir'])
tables_dir = out_dir / 'tables'
reports_dir = out_dir / 'reports'
plots_dir = out_dir / 'plots'

required_tables = [
    'fit_vs_complexity_summary.csv',
    'deconvolution_quality_summary.csv',
    'component_marker_profiles.csv',
    'component_marker_enrichment.csv',
    'deterministic_vs_probabilistic_summary.csv',
    'per_class_method_metrics.csv',
    'fit_vs_interpretability.csv',
    'k_selection_summary.csv',
    'class_component_means.csv',
    'per_cell_weight_entropy.csv',
]

status = []
for name in required_tables:
    path = tables_dir / name
    status.append({'table': name, 'exists': path.exists(), 'path': str(path)})
status_df = pd.DataFrame(status)
display(status_df)
print('All required tables present:', bool(status_df['exists'].all()))

In [ ]:
manifest_path = reports_dir / 'artifact_manifest.json'
if manifest_path.exists():
    import json
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('Manifest summary keys:', sorted(manifest.keys()))
    print('Number of tables:', len(manifest.get('tables', [])))
    print('Number of plots:', len(manifest.get('plots', [])))
else:
    print('Missing artifact manifest:', manifest_path)

In [ ]:
k_path = tables_dir / 'k_selection_summary.csv'
if k_path.exists():
    k_df = pd.read_csv(k_path)
    display(k_df.sort_values(['method', 'k']).head(30))
else:
    print('Missing:', k_path)